# 13.2 Node2Vec — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter13_2_node2vec.ipynb)

책 본문: [13.2 Node2Vec](https://smhanlab.com/book-ml/kor/ml1/chapter13/2.html)


이 노트북은 카라테 클럽 그래프 위에서 Node2Vec(무작위 걷기 → skip-gram)을
실제로 돌려봅니다.


In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
IMG = "/home/smhan/book-ml/kor/src/images"  # 그림 저장 위치(로컬). Colab에서는 이 줄을 수정하세요.

## 1. Zachary's Karate Club

34개 노드, 78개 엣지의 표준 그래프. 1977년 실제 분열: **감독파**(node 0, 파랑) vs **관장파**(node 33, 주황).
임베딩 학습에는 라벨을 주지 않되, 결과 그림에서만 색으로 구분해 "구조가 분열을 회복하는지" 확인합니다.

In [2]:
import random, math
import numpy as np

KARATE_EDGES = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),
    (0,12),(0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),
    (1,17),(1,19),(1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),
    (2,28),(2,32),(3,7),(3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),
    (6,16),(8,30),(8,32),(8,33),(9,33),(13,33),(14,32),(14,33),(15,32),
    (15,33),(18,32),(18,33),(19,33),(20,32),(20,33),(22,32),(22,33),
    (23,25),(23,27),(23,29),(23,32),(23,33),(24,25),(24,27),(24,31),
    (25,31),(26,29),(26,33),(27,33),(28,31),(28,33),(29,32),(29,33),
    (30,32),(30,33),(31,32),(31,33),(32,33),
]
N_NODES = 34
NB = {i: [] for i in range(N_NODES)}
for a, b in KARATE_EDGES:
    NB[a].append(b)
    NB[b].append(a)
FACTION = [0]*34  # 0 = 감독파, 1 = 관장파
for i in [9,14,15,18,20,22,23,24,25,26,27,28,29,30,33]:
    FACTION[i] = 1
print("노드 0(감독)의 도:", len(NB[0]), "  노드 33(관장)의 도:", len(NB[33]))

노드 0(감독)의 도: 16   노드 33(관장)의 도: 17


## 2. 무작위 걷기 = "문장"

각 노드에서 출발해 이웃을 무작위로 골라 길이 `walk_length`로 걷는다 — 이 걷기들이 곧 Node2Vec이 학습할 "말뭉치"다.
일괄적(uniform) 버전부터 보고, 다음 섹션에서 Node2Vec의 편향 버전(p, q 계수)으로 확장한다.

In [3]:
def random_walk(nb, start, length, rng):
    walk = [start]
    for _ in range(length - 1):
        cur = walk[-1]
        if not nb[cur]:
            break
        walk.append(rng.choice(nb[cur]))
    return walk

def build_walks(nb, n_nodes, walks_per_node=10, walk_length=8, seed=42):
    rng = random.Random(seed)
    walks = []
    for node in range(n_nodes):
        for _ in range(walks_per_node):
            walks.append(random_walk(nb, node, walk_length, rng))
    return walks

walks = build_walks(NB, N_NODES)
print("걷기 총 개수:", len(walks), "  예시 두 걷기:")
print(walks[0])
print(walks[-1])

걷기 총 개수: 340   예시 두 걷기:
[0, 4, 0, 10, 0, 8, 2, 1]
[33, 9, 2, 0, 1, 19, 0, 8]


## 3. Node2Vec: 걷기 말뭉치로 skip-gram 학습

14.2절의 `train_skipgram`과 **같은** 함수 — 토큰이 단어 대신 노드 번호일 뿐.
d=8차원 임베딩을 학습하고 SVD로 2차원으로 투영해 그린다(= 본문 그림의 재현).

In [4]:
def train_skipgram(corpus, window=2, dim=8, epochs=50, lr=0.05, neg_k=3, seed=42):
    rng = random.Random(seed)
    vocab = sorted(set(corpus))
    idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)
    W_in  = [[rng.uniform(-0.5, 0.5) for _ in range(dim)] for _ in range(V)]
    W_out = [[rng.uniform(-0.5, 0.5) for _ in range(dim)] for _ in range(V)]

    def sigmoid(z):
        return 1 / (1 + math.exp(-max(-20, min(20, z))))

    pairs = []
    for i, center in enumerate(corpus):
        for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
            if j != i:
                pairs.append((idx[center], idx[corpus[j]]))

    for _ in range(epochs):
        rng.shuffle(pairs)
        for c, o in pairs:
            targets = [(o, 1)] + [(rng.randrange(V), 0) for _ in range(neg_k)]
            for t, label in targets:
                z = sum(W_in[c][k] * W_out[t][k] for k in range(dim))
                pred = sigmoid(z)
                grad = (pred - label) * lr
                for k in range(dim):
                    g_in, g_out = W_in[c][k], W_out[t][k]
                    W_in[c][k]  -= grad * g_out
                    W_out[t][k] -= grad * g_in
    return {w: W_in[idx[w]] for w in vocab}

corpus = []
for w in walks:
    corpus += [str(n) for n in w]
emb = train_skipgram(corpus, window=2, dim=8, epochs=60, lr=0.05, neg_k=3, seed=42)
print("어휘(노드) 수:", len(emb))

def project2d(emb, nodes):
    X = np.array([emb[str(i)] for i in nodes])
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

Z = project2d(emb, range(N_NODES))

# --- 본문 그림 캡션 수치 검증: 각 리더가 자기 파벌 군집 중심에서 얼마나 가까운가 ---
_cents = {f: Z[[i for i in range(N_NODES) if FACTION[i] == f]].mean(axis=0) for f in (0, 1)}
def _d2(a, b): return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))
print(f"node 0(감독) -> 자기 파벌 중심 거리: {_d2(Z[0], _cents[0]):.2f}")
print(f"node 33(관장) -> 자기 파벌 중심 거리: {_d2(Z[33], _cents[1]):.2f}")
_o0 = sorted(_d2(Z[i], _cents[0]) for i in range(N_NODES) if FACTION[i] == 0 and i != 0)
_o1 = sorted(_d2(Z[i], _cents[1]) for i in range(N_NODES) if FACTION[i] == 1 and i != 33)
print(f"나머지 멤버의 자기 파벌 중심까지 중앙값: 감독파 {np.median(_o0):.2f}, 관장파 {np.median(_o1):.2f}")
cmap = {0: "tab:blue", 1: "tab:orange"}
plt.figure(figsize=(7, 6))
for f, lab in [(0, "Coach faction (node 0)"), (1, "Owner faction (node 33)")]:
    pts = Z[[i for i in range(N_NODES) if FACTION[i] == f]]
    plt.scatter(pts[:, 0], pts[:, 1], c=cmap[f], s=45, label=lab, alpha=0.85)
for n in (0, 33):
    plt.annotate(str(n), (Z[n, 0], Z[n, 1]), fontsize=13, fontweight="bold",
                 xytext=(6, 6), textcoords="offset points")
plt.title("Node2Vec embedding (p=q=1), 2D projection (SVD)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG + "/ch14_3_node2vec_karate.svg", bbox_inches="tight")
plt.show()

어휘(노드) 수: 34
node 0(감독) -> 자기 파벌 중심 거리: 0.22
node 33(관장) -> 자기 파벌 중심 거리: 0.26
나머지 멤버의 자기 파벌 중심까지 중앙값: 감독파 1.06, 관장파 0.44


## 4. Node2Vec의 p·q: 걷기 스타일을 바꾸는 두 계수

- p < 1: 방금 있던 노드로 **뒤로 감**이 쉬워진다
- q < 1: 방금 탐색한 로컬 영역에 **머물기** 쉬워진다 (BFS형, 논문의 (1,16) 프리셋)
- q > 1: 새 영토로 **확장**하려 한다 (DFS형, 논문의 (16,1) 프리셋)

스케일을 줄여 (1,10), (10,1)로, 임베딩의 파벌 분리 품질(코사인 실루엣)과 node 2의 상위 3개 유사 노드를 비교한다.

In [5]:
def biased_walk(nb, start, length, p, q, rng):
    walk = [start]
    prev = None
    for _ in range(length - 1):
        cur = walk[-1]
        weights = []
        for nxt in nb[cur]:
            if prev is None:
                weights.append(1.0)
            elif nxt == prev:
                weights.append(1.0 / p)      # 돌아가기
            elif nxt in nb[prev]:
                weights.append(1.0 / q)      # 로컬 영역 머무르기
            else:
                weights.append(1.0)          # 새 영토로
        r = rng.random() * sum(weights)
        acc, chosen = 0.0, nb[cur][0]
        for nxt, w in zip(nb[cur], weights):
            acc += w
            if r <= acc:
                chosen = nxt
                break
        walk.append(chosen)
        prev, cur = cur, chosen
    return walk

def gen_walks(walks_per_node, length, p, q, seed):
    rng = random.Random(seed)
    return [biased_walk(NB, node, length, p, q, rng)
            for node in range(N_NODES) for _ in range(walks_per_node)]

def cos_sim(a, b):
    d = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a)); nb_ = math.sqrt(sum(x*x for x in b))
    return d / (na*nb_ + 1e-9)

def silhouette_of(emb):
    from sklearn.metrics import silhouette_score
    X = np.array([emb[str(i)] for i in range(N_NODES)])
    return silhouette_score(X, np.array(FACTION), metric="cosine")

settings = [((1, 1), "Uniform p=q=1"), ((1, 10), "BFS-like p=1, q=10"), ((10, 1), "DFS-like p=10, q=1")]
results = []
for (p, q), name in settings:
    wk = gen_walks(10, 8, p, q, seed=100)
    corp = []
    for w in wk:
        corp += [str(n) for n in w]
    e = train_skipgram(corp, window=2, dim=8, epochs=60, lr=0.05, neg_k=3, seed=42)
    sil = silhouette_of(e)
    top3 = sorted(((w, cos_sim(e["2"], e[w])) for w in e if w != "2"),
                  key=lambda kv: -kv[1])[:3]
    results.append((name, sil, top3, e))
    print(f"{name}: 실루엣={sil:.3f}  node2 상위3={[(w, round(s, 3)) for w, s in top3]}")

Uniform p=q=1: 실루엣=0.351  node2 상위3=[('7', 0.731), ('13', 0.717), ('9', 0.716)]


BFS-like p=1, q=10: 실루엣=0.342  node2 상위3=[('9', 0.858), ('28', 0.73), ('7', 0.722)]


DFS-like p=10, q=1: 실루엣=0.393  node2 상위3=[('12', 0.827), ('19', 0.699), ('17', 0.651)]


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, sil, top3, e) in zip(axes, results):
    Zk = project2d(e, range(N_NODES))
    for f, lab in [(0, "Coach"), (1, "Owner")]:
        pts = Zk[[i for i in range(N_NODES) if FACTION[i] == f]]
        ax.scatter(pts[:, 0], pts[:, 1], c=cmap[f], s=30, alpha=0.85)
    ax.set_title(f"{name}\nsilhouette={sil:.3f}")
    ax.grid(alpha=0.3)
fig.suptitle("Node2Vec embeddings by p and q (2D projection)", y=1.03)
plt.tight_layout()
plt.savefig(IMG + "/ch14_3_pq_effects.svg", bbox_inches="tight")
plt.show()

## 정리

- **Node2Vec** = 무작위 걷기(문장) + skip-gram(14.2절) → 노드 임베딩. p·q 계수가 커뮤니티(DFS형) vs 역할(BFS형) 구조의 초점을 조절.
- 라벨 없이 구조만 봤는데도 실제 분열(커뮤니티)이 회복된다. 같은 걷기의 "정상분포(머무는 확률)"라는 다른 질문은 13.3절의 PageRank로 이어진다.
